## This notebook. is to add school district mapping

In [1]:
import pathlib
import geopandas as gpd
import pandas as pd

DATA_PATH = pathlib.Path.cwd().parent / "data" 
GEOJSON_PATH = DATA_PATH / "raw" / "DistrictAreas.geojson"
FILE_PATHS = {
    "listings": DATA_PATH / "processed" / "listing_validated.csv",
    "sold": DATA_PATH / "processed" / "sold_validated.csv",
}


gdf = gpd.read_file(GEOJSON_PATH)
listdf = pd.read_csv(FILE_PATHS["listings"])
solddf = pd.read_csv(FILE_PATHS["sold"])

/var/folders/dc/245wkwwd22jf9mtnyn_s6_pr0000gn/T/ipykernel_68824/4079380819.py:14: DtypeWarning: Columns (0: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  listdf = pd.read_csv(FILE_PATHS["listings"])


In [2]:
listgdf = gpd.GeoDataFrame(
    listdf, geometry=gpd.points_from_xy(listdf['Longitude'], listdf['Latitude']), crs="EPSG:4326"
)

solddf = gpd.GeoDataFrame(
    solddf, geometry=gpd.points_from_xy(solddf['Longitude'], solddf['Latitude']), crs="EPSG:4326"
)

Preprocessing District Areas

In [3]:
gdf = gdf[gdf["DistrictType"]=="Unified"]
gdf = gdf.set_index("DistrictName")
gdf["area"] = gdf["geometry"].area
gdf = gdf[["area", "geometry"]]

Joining dataframes

In [4]:
listgdf = listgdf.to_crs(gdf.crs)
solddf = solddf.to_crs(gdf.crs)

listgdf = gpd.sjoin(gdf, listgdf, how="right", predicate="contains")
soldgdf = gpd.sjoin(gdf, solddf, how="right", predicate="contains")

In [5]:
import os


listgdf.to_csv(DATA_PATH / "processed" / "listings_districted.csv", index=False)
soldgdf.to_csv(DATA_PATH / "processed" / "sold_districted.csv", index=False)

os.remove(FILE_PATHS["listings"])
os.remove(FILE_PATHS["sold"])